# 03 — Sistema Especialista baseado em Regras SE-ENTÃO

**Objetivo:** construir a base de conhecimento (regras SE-ENTÃO) separada do motor de inferência, medir o **esforço humano** com cronômetro e a **eficiência computacional apenas na inferência**.

**Protocolo:** regras elaboradas **exclusivamente com o conjunto de treino**. As métricas finais são calculadas no teste.

In [1]:
import sys, json, os
sys.path.append('..')

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import confusion_matrix

from src.config import (
    SEED, TEST_SIZE, CLASSIFICATION_TARGET, REGRESSION_TARGET,
    RISK_CLASSES, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, RULES_DIR, LOGS_DIR,
)
from src.data import split_train_test, encode_target
from src.expert_system import (
    Condition, Rule, KnowledgeBase, InferenceEngine, RuleTimer,
    summarize_timer_log,
)
from src import metrics, profiling

np.random.seed(SEED)
pd.set_option('display.max_columns', 60)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
(REPORTS_DIR / 'predictions').mkdir(parents=True, exist_ok=True)
(REPORTS_DIR / 'models').mkdir(parents=True, exist_ok=True)
print('Configuração carregada. Semente =', SEED)

Configuração carregada. Semente = 42


## 1) Carregamento e divisão treino/teste

Mesma divisão usada por todos os modelos: 80/20, `random_state=42`, estratificada na classificação. As colunas de vazamento (`Burnout_Score`, `Burnout_Risk`, `Mental_Health_Status`, `AI_Wellness_Recommendation`, `Person_ID`) já foram removidas em `02_preprocessing`.

In [2]:
df_reg = pd.read_csv('../data/processed/dataset_regression.csv')
df_clf = pd.read_csv('../data/processed/dataset_classification.csv')

Xr_train, Xr_test, yr_train, yr_test = split_train_test(df_reg, REGRESSION_TARGET, stratify=False)
Xc_train, Xc_test, yc_train_raw, yc_test_raw = split_train_test(df_clf, CLASSIFICATION_TARGET, stratify=True)

yc_train = encode_target(yc_train_raw)
yc_test = encode_target(yc_test_raw)
yc_train.index = Xc_train.index
yc_test.index = Xc_test.index

print('Regressão  — treino:', Xr_train.shape, '| teste:', Xr_test.shape)
print('Classificação — treino:', Xc_train.shape, '| teste:', Xc_test.shape)
print('\nDistribuição do alvo de classificação (treino):')
print(yc_train_raw.value_counts(normalize=True).round(4))

Regressão  — treino: (40000, 85) | teste: (10000, 85)
Classificação — treino: (40000, 85) | teste: (10000, 85)

Distribuição do alvo de classificação (treino):
Burnout_Risk
Low         0.5074
Moderate    0.2956
High        0.1970
Name: proportion, dtype: float64


## 2) Conhecimento extraído **apenas do treino**

As regras não são inventadas no vácuo: os limiares vêm de quantis e de médias condicionais calculadas no treino. O teste não é consultado nesta etapa.

In [3]:
# Correlação das variáveis numéricas com o alvo (somente treino)
corr_reg = (
    Xr_train.select_dtypes(include=np.number)
    .corrwith(yr_train)
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .rename('correlacao_com_Burnout_Score')
    .to_frame()
)
corr_reg.round(4).to_csv(TABLES_DIR / '03_correlacoes_treino.csv')
corr_reg.head(15).round(3)

,correlacao_com_Burnout_Score
Anxiety_Score,0.917
Emotional_Stability,-0.909
Mood_Score,-0.908
Depression_Score,0.903
Productivity_Score,-0.885
Stress_Level,0.867
Chronic_Stress,0.628
Work_Hours_Per_Week,0.605
Screen_Time_Hours,0.347
Coffee_Cups_Per_Day,0.269


In [4]:
# Quantis das variáveis usadas nas regras (somente treino)
features_regras = [
    'Anxiety_Score', 'Depression_Score', 'Mood_Score', 'Emotional_Stability',
    'Productivity_Score', 'Work_Hours_Per_Week', 'Stress_Level', 'Chronic_Stress',
    'Sleep_Hours', 'Sleep_Quality', 'Physical_Activity_Hours', 'Life_Satisfaction',
    'Meditation_Minutes',
]
quantis = Xr_train[features_regras].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).T
quantis.columns = ['p10', 'p25', 'p50', 'p75', 'p90']
quantis.round(2).to_csv(TABLES_DIR / '03_quantis_treino.csv')
quantis.round(2)

,p10,p25,p50,p75,p90
Anxiety_Score,5.0,20.0,39.0,59.0,76.0
Depression_Score,5.0,20.0,39.0,59.0,77.0
Mood_Score,24.0,42.0,61.0,80.0,95.0
Emotional_Stability,23.0,40.0,60.0,81.0,94.0
Productivity_Score,23.0,41.0,61.0,80.0,94.0
Work_Hours_Per_Week,25.0,32.0,45.0,58.0,65.0
Stress_Level,0.0,0.0,1.0,1.0,2.0
Chronic_Stress,0.0,0.0,0.0,0.0,1.0
Sleep_Hours,4.6,5.5,7.0,8.5,9.4
Sleep_Quality,0.0,1.0,2.0,2.0,3.0


In [5]:
# Médias condicionais por classe de risco (somente treino)
base_clf = Xc_train.copy()
base_clf['Burnout_Risk'] = yc_train_raw.values

perfil_classes = base_clf.groupby('Burnout_Risk')[
    ['Anxiety_Score', 'Depression_Score', 'Mood_Score', 'Emotional_Stability',
     'Productivity_Score', 'Stress_Level', 'Work_Hours_Per_Week', 'Sleep_Hours']
].mean()
perfil_classes.round(2).to_csv(TABLES_DIR / '03_perfil_classes_treino.csv')
perfil_classes.round(2)

,Anxiety_Score,Depression_Score,Mood_Score,Emotional_Stability,Productivity_Score,Stress_Level,Work_Hours_Per_Week,Sleep_Hours
Burnout_Risk,,,,,,,,
High,75.47,75.65,25.24,24.10,25.35,1.66,57.77,6.45
Low,20.90,21.11,78.79,79.04,78.35,0.20,37.44,7.29
Moderate,50.52,50.57,49.80,49.23,49.84,0.97,49.26,6.85


## 3) Base de conhecimento — Classificação (19 regras)

Tipos: **relação**, **recomendação**, **diretriz**, **estratégia** e **heurística**. Cada regra usa apenas conjunções **E** (nunca mistura E/OU). Os pesos funcionam como fatores de certeza: o motor soma os votos de todas as regras disparadas e escolhe a classe com maior pontuação.

In [6]:
# Limiares arredondados a partir dos quantis do treino
t = {
    'ans_alto': 75.0, 'ans_elevado': 60.0,
    'dep_alto': 75.0, 'dep_elevado': 60.0,
    'mood_baixo': 40.0, 'mood_muito_baixo': 25.0,
    'emo_baixo': 40.0, 'emo_muito_baixo': 25.0,
    'prod_baixa': 40.0, 'prod_muito_baixa': 25.0,
    'horas_alta': 58.0, 'horas_media': 45.0,
    'sono_bom': 7.0, 'atividade_boa': 7.0, 'satisfacao_alta': 8.0,
}

def C(feature, operator, value):
    return Condition(feature, operator, float(value))

regras_clf = [
    Rule(1, 'classificacao', 'heurística', [C('Anxiety_Score', '>=', t['ans_alto']), C('Depression_Score', '>=', t['dep_alto'])], 'High', 3.0, 'Ansiedade e depressão muito elevadas: combinação clássica de esgotamento.'),
    Rule(2, 'classificacao', 'relação', [C('Anxiety_Score', '>=', t['ans_elevado']), C('Depression_Score', '>=', t['dep_elevado'])], 'High', 2.0, 'Sintomas afetivos elevados associados a risco alto no treino.'),
    Rule(3, 'classificacao', 'relação', [C('Mood_Score', '<=', t['mood_baixo']), C('Emotional_Stability', '<=', t['emo_baixo'])], 'High', 2.0, 'Humor e estabilidade emocional baixos reduzem a capacidade de regulação.'),
    Rule(4, 'classificacao', 'estratégia', [C('Productivity_Score', '<=', t['prod_baixa']), C('Stress_Level', '>=', 2)], 'High', 2.0, 'Queda de produtividade sob estresse máximo indica exaustão.'),
    Rule(5, 'classificacao', 'estratégia', [C('Work_Hours_Per_Week', '>=', t['horas_alta']), C('Stress_Level', '>=', 2)], 'High', 1.5, 'Sobrecarga horária somada a estresse intenso.'),
    Rule(6, 'classificacao', 'heurística', [C('Chronic_Stress', '>=', 1), C('Anxiety_Score', '>=', t['ans_elevado'])], 'High', 1.5, 'Estresse crônico potencializa ansiedade elevada.'),
    Rule(7, 'classificacao', 'relação', [C('Anxiety_Score', '>=', t['ans_alto'])], 'High', 1.0, 'Ansiedade muito alta isoladamente já eleva o risco.'),
    Rule(8, 'classificacao', 'relação', [C('Depression_Score', '>=', t['dep_alto'])], 'High', 1.0, 'Depressão muito alta isoladamente já eleva o risco.'),
    Rule(9, 'classificacao', 'diretriz', [C('Mood_Score', '<=', t['mood_muito_baixo'])], 'High', 1.5, 'Humor muito baixo é indicador de gravidade.'),
    Rule(10, 'classificacao', 'diretriz', [C('Emotional_Stability', '<=', t['emo_muito_baixo'])], 'High', 1.5, 'Instabilidade emocional severa.'),
    Rule(11, 'classificacao', 'diretriz', [C('Productivity_Score', '<=', t['prod_muito_baixa'])], 'High', 1.0, 'Produtividade muito reduzida.'),
    Rule(12, 'classificacao', 'relação', [C('Anxiety_Score', '>=', 40), C('Anxiety_Score', '<', t['ans_elevado'])], 'Moderate', 1.0, 'Ansiedade intermediária: risco moderado.'),
    Rule(13, 'classificacao', 'relação', [C('Depression_Score', '>=', 40), C('Depression_Score', '<', t['dep_elevado'])], 'Moderate', 1.0, 'Depressão intermediária: risco moderado.'),
    Rule(14, 'classificacao', 'heurística', [C('Mood_Score', '>', t['mood_baixo']), C('Mood_Score', '<=', 60)], 'Moderate', 1.0, 'Humor na faixa intermediária.'),
    Rule(15, 'classificacao', 'heurística', [C('Emotional_Stability', '>', t['emo_baixo']), C('Emotional_Stability', '<=', 60)], 'Moderate', 1.0, 'Estabilidade emocional na faixa intermediária.'),
    Rule(16, 'classificacao', 'diretriz', [C('Work_Hours_Per_Week', '>=', t['horas_media'])], 'Moderate', 0.5, 'Jornada acima da mediana aumenta a exposição.'),
    Rule(17, 'classificacao', 'estratégia', [C('Stress_Level', '==', 1)], 'Moderate', 0.5, 'Estresse moderado (nível 1).'),
    Rule(18, 'classificacao', 'recomendação', [C('Sleep_Hours', '>=', t['sono_bom']), C('Physical_Activity_Hours', '>=', t['atividade_boa']), C('Life_Satisfaction', '>=', t['satisfacao_alta'])], 'Low', 2.0, 'Hábitos protetores (sono, atividade física e satisfação) reduzem o risco.'),
    Rule(19, 'classificacao', 'recomendação', [C('Mood_Score', '>=', 60), C('Emotional_Stability', '>=', 60), C('Anxiety_Score', '<', 40)], 'Low', 1.5, 'Perfil emocional saudável sustenta classificação de baixo risco.'),
]

kb = KnowledgeBase(regras_clf)
engine = InferenceEngine(kb)
print(f'Base de classificação: {len(kb.by_problem("classificacao"))} regras ativas')

Base de classificação: 19 regras ativas


In [7]:
# Suporte (cobertura) de cada regra no treino — evita regras que nunca disparam
suporte_clf = engine.rule_firing_counts(Xc_train, 'classificacao')
suporte_clf.to_csv(TABLES_DIR / '03_suporte_regras_classificacao.csv', index=False)
suporte_clf

,regra,tipo,antecedente,consequente,peso,disparos,cobertura (%)
0,REGRA-01,heurística,Anxiety_Score >= 75 AND Depression_Score >= 75,High,3.0,3226,8.065
1,REGRA-02,relação,Anxiety_Score >= 60 AND Depression_Score >= 60,High,2.0,7611,19.028
2,REGRA-03,relação,Mood_Score <= 40 AND Emotional_Stability <= 40,High,2.0,7865,19.662
3,REGRA-04,estratégia,Productivity_Score <= 40 AND Stress_Level >= 2,High,2.0,5437,13.592
4,REGRA-05,estratégia,Work_Hours_Per_Week >= 58 AND Stress_Level >= 2,High,1.5,3801,9.502
5,REGRA-06,heurística,Chronic_Stress >= 1 AND Anxiety_Score >= 60,High,1.5,5306,13.265
6,REGRA-07,relação,Anxiety_Score >= 75,High,1.0,4433,11.083
7,REGRA-08,relação,Depression_Score >= 75,High,1.0,4614,11.535
8,REGRA-09,diretriz,Mood_Score <= 25,High,1.5,4342,10.855
9,REGRA-10,diretriz,Emotional_Stability <= 25,High,1.5,4621,11.552


## 4) Base de conhecimento — Regressão (16 regras)

Na regressão cada regra contribui com um **delta em pontos** de `Burnout_Score`. Os deltas são heurísticos e monotônicos (quanto pior o indicador, maior o acréscimo). O escore final é uma reta de calibração ajustada **no treino** sobre a soma dos deltas.

In [8]:
regras_reg = [
    Rule(1, 'regressao', 'relação', [C('Anxiety_Score', '>=', 75)], 'Burnout_Score', 8.0, 'Ansiedade muito alta aumenta o escore.'),
    Rule(2, 'regressao', 'relação', [C('Anxiety_Score', '>=', 60), C('Anxiety_Score', '<', 75)], 'Burnout_Score', 5.0, 'Ansiedade elevada em faixa intermediária.'),
    Rule(3, 'regressao', 'relação', [C('Depression_Score', '>=', 75)], 'Burnout_Score', 8.0, 'Depressão muito alta aumenta o escore.'),
    Rule(4, 'regressao', 'relação', [C('Depression_Score', '>=', 60), C('Depression_Score', '<', 75)], 'Burnout_Score', 5.0, 'Depressão elevada em faixa intermediária.'),
    Rule(5, 'regressao', 'relação', [C('Mood_Score', '<=', 25)], 'Burnout_Score', 6.0, 'Humor muito baixo eleva o escore.'),
    Rule(6, 'regressao', 'relação', [C('Mood_Score', '>', 25), C('Mood_Score', '<=', 40)], 'Burnout_Score', 4.0, 'Humor baixo em faixa intermediária.'),
    Rule(7, 'regressao', 'relação', [C('Emotional_Stability', '<=', 25)], 'Burnout_Score', 7.0, 'Instabilidade emocional severa.'),
    Rule(8, 'regressao', 'relação', [C('Emotional_Stability', '>', 25), C('Emotional_Stability', '<=', 40)], 'Burnout_Score', 4.0, 'Instabilidade emocional moderada.'),
    Rule(9, 'regressao', 'estratégia', [C('Productivity_Score', '<=', 25)], 'Burnout_Score', 5.0, 'Produtividade muito reduzida.'),
    Rule(10, 'regressao', 'estratégia', [C('Productivity_Score', '>', 25), C('Productivity_Score', '<=', 40)], 'Burnout_Score', 3.0, 'Produtividade reduzida.'),
    Rule(11, 'regressao', 'heurística', [C('Work_Hours_Per_Week', '>=', 58)], 'Burnout_Score', 5.0, 'Jornada muito longa.'),
    Rule(12, 'regressao', 'heurística', [C('Work_Hours_Per_Week', '>=', 50), C('Work_Hours_Per_Week', '<', 58)], 'Burnout_Score', 2.0, 'Jornada acima do padrão.'),
    Rule(13, 'regressao', 'diretriz', [C('Stress_Level', '>=', 2)], 'Burnout_Score', 4.0, 'Estresse máximo autorreferido.'),
    Rule(14, 'regressao', 'diretriz', [C('Chronic_Stress', '>=', 1)], 'Burnout_Score', 4.0, 'Estresse crônico autorreferido.'),
    Rule(15, 'regressao', 'estratégia', [C('Sleep_Hours', '<', 6), C('Physical_Activity_Hours', '<', 3)], 'Burnout_Score', 5.0, 'Privação de sono combinada com sedentarismo.'),
    Rule(16, 'regressao', 'recomendação', [C('Sleep_Hours', '>=', 8), C('Physical_Activity_Hours', '>=', 7), C('Meditation_Minutes', '>=', 45)], 'Burnout_Score', -8.0, 'Conjunto de hábitos protetores reduz o escore.'),
]
for r in regras_reg:
    kb.add_rule(r)

engine = InferenceEngine(kb)
print(f'Base de regressão: {len(kb.by_problem("regressao"))} regras ativas')
print(f'Total na base de conhecimento: {len(kb.active_rules())} regras')

Base de regressão: 16 regras ativas
Total na base de conhecimento: 35 regras


In [9]:
suporte_reg = engine.rule_firing_counts(Xr_train, 'regressao')
suporte_reg.to_csv(TABLES_DIR / '03_suporte_regras_regressao.csv', index=False)
suporte_reg

,regra,tipo,antecedente,consequente,peso,disparos,cobertura (%)
0,REGRA-01,relação,Anxiety_Score >= 75,Burnout_Score,8.0,4443,11.108
1,REGRA-02,relação,Anxiety_Score >= 60 AND Anxiety_Score < 75,Burnout_Score,5.0,5240,13.100
2,REGRA-03,relação,Depression_Score >= 75,Burnout_Score,8.0,4614,11.535
3,REGRA-04,relação,Depression_Score >= 60 AND Depression_Score < 75,Burnout_Score,5.0,5238,13.095
4,REGRA-05,relação,Mood_Score <= 25,Burnout_Score,6.0,4345,10.862
5,REGRA-06,relação,Mood_Score > 25 AND Mood_Score <= 40,Burnout_Score,4.0,5195,12.987
6,REGRA-07,relação,Emotional_Stability <= 25,Burnout_Score,7.0,4631,11.578
7,REGRA-08,relação,Emotional_Stability > 25 AND Emotional_Stabili...,Burnout_Score,4.0,5398,13.495
8,REGRA-09,estratégia,Productivity_Score <= 25,Burnout_Score,5.0,4583,11.458
9,REGRA-10,estratégia,Productivity_Score > 25 AND Productivity_Score...,Burnout_Score,3.0,5154,12.885


## 5) Calibração da regressão e persistência da base

A soma dos deltas é uma escala arbitrária. Ajusta-se, **no treino**, uma reta `Burnout_Score = intercepto + inclinação × delta`. Essa calibração é parte da aquisição de conhecimento (não é um modelo preditivo aprendido: são apenas dois parâmetros de escala documentados).

In [10]:
delta_train = engine.regression_delta(Xr_train).reshape(-1, 1)

calibrador = LinearRegression().fit(delta_train, yr_train)
kb.regression_calibration = {
    'intercept': float(calibrador.intercept_),
    'slope': float(calibrador.coef_[0]),
}
engine = InferenceEngine(kb)
print('Calibração: Burnout_Score = {:.4f} + {:.4f} * delta'.format(
    kb.regression_calibration['intercept'], kb.regression_calibration['slope']))

# Persistência: CSV estruturado (por problema), texto legível e metadados
kb_clf_only = KnowledgeBase([r for r in kb.rules if r.problem == 'classificacao'])
kb_reg_only = KnowledgeBase([r for r in kb.rules if r.problem == 'regressao'])
kb_clf_only.save(RULES_DIR / 'kb_classificacao.csv')
kb_reg_only.save(RULES_DIR / 'kb_regressao.csv')
kb.save(RULES_DIR / 'kb_completa.csv')
kb.save_metadata(RULES_DIR / 'kb_metadata.json')

kb_clf_only.save_text(RULES_DIR / 'regras_classificacao.txt', 'Classificação (Burnout_Risk)')
kb_reg_only.save_text(RULES_DIR / 'regras_regressao.txt', 'Regressão (Burnout_Score)')
print('Arquivos salvos em rules/')

Calibração: Burnout_Score = 26.9892 + 1.3899 * delta
Arquivos salvos em rules/


## 6) Avaliação no treino e no teste

Métricas da Aula 08. O especialista **não é treinado**: as regras e a calibração são fixadas e aplicadas a ambos os conjuntos. O *gap* mede apenas a estabilidade das regras entre treino e teste.

In [11]:
def avaliar_classificacao(nome, y_true, y_pred, y_proba):
    m = metrics.classification_metrics(y_true, y_pred, y_proba=y_proba, classes=[0, 1, 2])
    m.update({'Modelo': nome, 'Problema': 'Classificacao'})
    return m

def avaliar_regressao(nome, y_true, y_pred):
    m = metrics.regression_metrics(y_true, y_pred)
    m.update({'Modelo': nome, 'Problema': 'Regressao'})
    return m

# Classificação
pred_c_train = engine.predict_classification(Xc_train)
pred_c_test = engine.predict_classification(Xc_test)
proba_c_train = engine.predict_proba_classification(Xc_train)
proba_c_test = engine.predict_proba_classification(Xc_test)

m_c_train = avaliar_classificacao('Sistema Especialista', yc_train.values, pred_c_train, proba_c_train)
m_c_test = avaliar_classificacao('Sistema Especialista', yc_test.values, pred_c_test, proba_c_test)

# Regressão
pred_r_train = engine.predict_regression(Xr_train)
pred_r_test = engine.predict_regression(Xr_test)
m_r_train = avaliar_regressao('Sistema Especialista', yr_train.values, pred_r_train)
m_r_test = avaliar_regressao('Sistema Especialista', yr_test.values, pred_r_test)

resumo_clf = pd.DataFrame([m_c_train, m_c_test], index=['Treino', 'Teste'])
resumo_reg = pd.DataFrame([m_r_train, m_r_test], index=['Treino', 'Teste'])
print('CLASSIFICAÇÃO'); display(resumo_clf.round(4))
print('REGRESSÃO'); display(resumo_reg.round(4))

CLASSIFICAÇÃO


,Accuracy,Precision (macro),Recall (macro),F1 (macro),Precision (weighted),Recall (weighted),F1 (weighted),ROC-AUC (OVR macro),PR-AUC (macro),Modelo,Problema
Treino,0.7920,0.7837,0.7914,0.7829,0.8152,0.7920,0.7984,0.8844,0.7726,Sistema Especialista,Classificacao
Teste,0.7987,0.7913,0.7986,0.7907,0.8199,0.7987,0.8046,0.8884,0.7796,Sistema Especialista,Classificacao


REGRESSÃO


,MAE,MSE,RMSE,MAPE (%),R²,MedAE,Cobertura MAPE (%),Modelo,Problema
Treino,13.0963,250.2224,15.8184,85.1674,0.6358,11.9749,93.11,Sistema Especialista,Regressao
Teste,13.1626,250.6768,15.8328,92.5491,0.6367,11.9892,93.00,Sistema Especialista,Regressao


In [12]:
gap_clf = metrics.generalization_gap(m_c_train, m_c_test)
gap_reg = metrics.generalization_gap(m_r_train, m_r_test)
print('Gap de generalização — classificação (treino - teste):')
print({k: round(v, 4) for k, v in gap_clf.items() if k in ['Gap F1 (macro)', 'Gap Accuracy', 'Gap ROC-AUC (OVR macro)']})
print('\nGap de generalização — regressão (treino - teste):')
print({k: round(v, 4) for k, v in gap_reg.items() if k in ['Gap RMSE', 'Gap MAE', 'Gap R²']})

Gap de generalização — classificação (treino - teste):
{'Gap Accuracy': -0.0067, 'Gap F1 (macro)': -0.0078, 'Gap ROC-AUC (OVR macro)': -0.0041}

Gap de generalização — regressão (treino - teste):
{'Gap MAE': -0.0662, 'Gap RMSE': -0.0144, 'Gap R²': -0.0009}


In [13]:
# Matriz de confusão e relatório por classe no teste
cm = metrics.confusion_matrix_frame(yc_test.values, pred_c_test, classes=RISK_CLASSES)
per_class = metrics.per_class_report(yc_test.values, pred_c_test, classes=RISK_CLASSES)
display(cm)
display(per_class.round(4))
cm.to_csv(TABLES_DIR / '03_matriz_confusao_especialista.csv', index=False)
per_class.to_csv(TABLES_DIR / '03_por_classe_especialista.csv', index=False)

,,Prev. Low,Prev. Moderate,Prev. High
0,Real Low,4088,986,0
1,Real Moderate,280,2298,378
2,Real High,5,364,1601


,Classe,Suporte,Precision,Recall,F1
0,Low,5074,0.9348,0.8057,0.8655
1,Moderate,2956,0.6299,0.7774,0.6959
2,High,1970,0.8090,0.8127,0.8108


## 7) Explicabilidade: rastreamento das regras disparadas

O motor registra quais regras dispararam para cada observação. Abaixo, um exemplo de cada classe de risco no teste.

In [14]:
exemplos = {}
for classe, codigo in [('Low', 0), ('Moderate', 1), ('High', 2)]:
    idx = int(np.where(yc_test.values == codigo)[0][0])
    linha = Xc_test.iloc[idx]
    disparos = engine.fired_rules(linha, 'classificacao')
    exemplos[classe] = {
        'indice_teste': idx,
        'real': classe,
        'previsto': RISK_CLASSES[int(pred_c_test[idx])],
        'regras_disparadas': [d['regra'] for d in disparos],
        'conclusoes': [d['entao'] for d in disparos],
    }
    print(f"\n[{classe}] índice {idx} -> previsto {RISK_CLASSES[int(pred_c_test[idx])]}")
    for d in disparos:
        print(f"  {d['regra']} ({d['tipo']}) SE {d['se']} ENTÃO {d['entao']} (peso {d['peso']})")

(REPORTS_DIR / 'predictions').mkdir(parents=True, exist_ok=True)
pd.Series(exemplos, dtype=object).to_json(
    REPORTS_DIR / 'predictions' / '03_exemplos_explicacao.json', force_ascii=False, indent=2
)


[Low] índice 0 -> previsto Moderate
  REGRA-15 (heurística) SE Emotional_Stability > 40 AND Emotional_Stability <= 60 ENTÃO Moderate (peso 1.0)
  REGRA-16 (diretriz) SE Work_Hours_Per_Week >= 45 ENTÃO Moderate (peso 0.5)

[Moderate] índice 1 -> previsto Moderate
  REGRA-13 (relação) SE Depression_Score >= 40 AND Depression_Score < 60 ENTÃO Moderate (peso 1.0)
  REGRA-14 (heurística) SE Mood_Score > 40 AND Mood_Score <= 60 ENTÃO Moderate (peso 1.0)
  REGRA-16 (diretriz) SE Work_Hours_Per_Week >= 45 ENTÃO Moderate (peso 0.5)
  REGRA-17 (estratégia) SE Stress_Level == 1 ENTÃO Moderate (peso 0.5)

[High] índice 9 -> previsto High
  REGRA-02 (relação) SE Anxiety_Score >= 60 AND Depression_Score >= 60 ENTÃO High (peso 2.0)
  REGRA-03 (relação) SE Mood_Score <= 40 AND Emotional_Stability <= 40 ENTÃO High (peso 2.0)
  REGRA-04 (estratégia) SE Productivity_Score <= 40 AND Stress_Level >= 2 ENTÃO High (peso 2.0)
  REGRA-06 (heurística) SE Chronic_Stress >= 1 AND Anxiety_Score >= 60 ENTÃO High (

## 8) Esforço humano (cronômetro) — separado do tempo computacional

O tempo de elaboração foi registrado em `logs/cronometro_regras.csv`, com **3 sessões para cada problema**. O notebook apenas agrega: média, desvio-padrão e tempo médio por regra.

In [15]:
cronometro = pd.read_csv(LOGS_DIR / 'cronometro_regras.csv')
display(cronometro)

resumo_humano = summarize_timer_log(LOGS_DIR / 'cronometro_regras.csv')
resumo_humano.to_csv(TABLES_DIR / '03_esforco_humano.csv', index=False)
display(resumo_humano)

print('Esforço humano total: {:.1f} minutos'.format(cronometro['tempo_total_min'].sum()))
print('Regras no arquivo:', len(kb.active_rules()), '| regras criadas registradas:', int(cronometro['n_regras_criadas'].sum()))

,data,horario_inicio,horario_fim,interrupcoes,tempo_total_min,n_regras_criadas,problema,tempo_medio_por_regra_min,observacoes
0,2026-10-21,09:00,09:45,nenhuma,45.0,6,classificacao,7.5000,Elaboração inicial com base em correlações do ...
1,2026-10-21,15:00,15:38,nenhuma,38.0,6,classificacao,6.3333,Refinamento por grupos de risco
2,2026-10-22,09:00,09:44,breve,44.0,7,classificacao,6.2857,Revisão e testes com regras no treino
3,2026-10-21,11:00,11:40,nenhuma,40.0,5,regressao,8.0000,Regras por faixas com base em correlações
4,2026-10-22,14:00,14:36,nenhuma,36.0,5,regressao,7.2000,Ajuste de pesos heurísticos
5,2026-10-22,16:00,16:39,nenhuma,39.0,6,regressao,6.5000,Consolidação e validação das regras no treino


,Problema,Sessões,Tempo total (min),Média por sessão (min),Desvio-padrão (min),Regras,Tempo médio por regra (min)
0,classificacao,3,127.0,42.33,3.79,19,6.68
1,regressao,3,115.0,38.33,2.08,16,7.19


Esforço humano total: 242.0 minutos
Regras no arquivo: 35 | regras criadas registradas: 35


> **Observação metodológica:** o cronômetro mede o esforço humano de elaboração/refinamento das regras. Ele **não** é somado aos tempos de CPU medidos abaixo.

## 9) Eficiência computacional — somente inferência

O sistema especialista não possui etapa de treinamento estatístico. Mede-se apenas a inferência, com 30 repetições (`time.perf_counter` + `tracemalloc`).

In [16]:
perfil_c = profiling.profile_inference(lambda X: engine.predict_classification(X), Xc_test, n_rep=30)
perfil_r = profiling.profile_inference(lambda X: engine.predict_regression(X), Xr_test, n_rep=30)

eficiencia = pd.DataFrame([
    profiling.format_profile_row('Sistema Especialista', 'Regras SE-ENTAO', perfil_c),
    profiling.format_profile_row('Sistema Especialista', 'Regras SE-ENTAO', perfil_r),
])
eficiencia.insert(1, 'Problema', ['Classificacao', 'Regressao'])
eficiencia.to_csv(TABLES_DIR / '03_eficiencia_especialista.csv', index=False)
eficiencia

,Família,Problema,Modelo,Tempo treino (s),Tempo inferência (média ± dp) s,Tempo por amostra (ms),Vazão (amostras/s),Pico memória inferência (KB),Pico memória treino (KB)
0,Sistema Especialista,Classificacao,Regras SE-ENTAO,Não se aplica,0.090848 ± 0.031166,0.009085,110073.49,496.34,Não se aplica
1,Sistema Especialista,Regressao,Regras SE-ENTAO,Não se aplica,0.023183 ± 0.006513,0.002318,431343.49,238.66,Não se aplica


## 10) Resultados consolidados do especialista

Salva-se um CSV com métricas de treino/teste e outro com as predições, para que os notebooks 06 e 07 reutilizem exatamente os mesmos números (sem reexecutar as regras).

In [17]:
linhas = []
for split, mc, mr in [('Treino', m_c_train, m_r_train), ('Teste', m_c_test, m_r_test)]:
    for problema, m in [('Classificacao', mc), ('Regressao', mr)]:
        linha = {'Familia': 'Sistema Especialista', 'Modelo': 'Regras SE-ENTAO',
                 'Problema': problema, 'Conjunto': split}
        linha.update({k: v for k, v in m.items() if k not in ['Modelo', 'Problema']})
        linhas.append(linha)

resultados = pd.DataFrame(linhas)
resultados.to_csv(TABLES_DIR / 'resultados_especialista.csv', index=False)

np.savez(REPORTS_DIR / 'predictions' / 'expert_classificacao.npz',
         y_true_test=yc_test.values, y_pred_test=pred_c_test, y_proba_test=proba_c_test,
         y_true_train=yc_train.values, y_pred_train=pred_c_train, y_proba_train=proba_c_train)
np.savez(REPORTS_DIR / 'predictions' / 'expert_regressao.npz',
         y_true_test=yr_test.values, y_pred_test=pred_r_test,
         y_true_train=yr_train.values, y_pred_train=pred_r_train)

# Metadados do "modelo" especialista (para 07)
meta = {
    'n_regras_classificacao': len(kb.by_problem('classificacao')),
    'n_regras_regressao': len(kb.by_problem('regressao')),
    'calibracao_regressao': kb.regression_calibration,
    'esforco_humano_min': float(cronometro['tempo_total_min'].sum()),
}
(REPORTS_DIR / 'models' / 'expert_meta.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')
print(resultados.round(4).to_string(index=False))

             Familia          Modelo      Problema Conjunto  Accuracy  Precision (macro)  Recall (macro)  F1 (macro)  Precision (weighted)  Recall (weighted)  F1 (weighted)  ROC-AUC (OVR macro)  PR-AUC (macro)     MAE      MSE    RMSE  MAPE (%)     R²   MedAE  Cobertura MAPE (%)
Sistema Especialista Regras SE-ENTAO Classificacao   Treino    0.7920             0.7837          0.7914      0.7829                0.8152             0.7920         0.7984               0.8844          0.7726     NaN      NaN     NaN       NaN    NaN     NaN                 NaN
Sistema Especialista Regras SE-ENTAO     Regressao   Treino       NaN                NaN             NaN         NaN                   NaN                NaN            NaN                  NaN             NaN 13.0963 250.2224 15.8184   85.1674 0.6358 11.9749               93.11
Sistema Especialista Regras SE-ENTAO Classificacao    Teste    0.7987             0.7913          0.7986      0.7907                0.8199             0.7987   

## 11) Conclusões do notebook

1. A base tem **19 regras de classificação** e **16 de regressão**, separada do motor de inferência.
2. As regras usam apenas o treino para limiares e calibração.
3. O especialista é totalmente auditável: cada predição pode ser explicada pelas regras disparadas.
4. O esforço humano (cronômetro) é reportado separadamente do custo computacional.
5. As predições e métricas ficam salvas para as etapas seguintes.